# DefectosCafeVerde — DVSR1 frozen validation screen

Applies the passed train-only reliability selector exactly once to validation. No selector fitting, detector/fuser training, or test access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()

roots=[Path('/content/drive/MyDrive'),Path('/content/drive/.shortcut-targets-by-id')]
audit_name='dvsr_routability.json'
matches=sorted({p for root in roots if root.exists() for p in root.rglob(audit_name) if p.is_file() and p.parent.name=='defectoscafeverde-dvsr-routability-v1'})
if not matches: raise FileNotFoundError(f'Tidak menemukan {audit_name}')
by_hash={}
for path in matches: by_hash.setdefault(sha256(path),[]).append(path)
if len(by_hash)!=1: raise RuntimeError(f'Ditemukan audit DVSR berbeda: {by_hash}')
aliases=next(iter(by_hash.values())); preferred=[p for p in aliases if '/MyDrive/' in p.as_posix()]
TRAIN_AUDIT=(preferred or aliases)[0]; PROJECT=TRAIN_AUDIT.parents[2]
CONFIRM=PROJECT/'experiments/defectoscafeverde-dvf1-confirmation-v1'
CONFIRMATION_ARM=CONFIRM/'DVF1_seed123_result.json'
CONFIRMATION_DECISION=CONFIRM/'DVF1_three_seed_confirmation.json'
VALIDATION_CACHE=PROJECT/'experiments/defectoscafeverde-dvf1-v1/val_pair_cache.pt'
OUTPUT=PROJECT/'experiments/defectoscafeverde-dvsr1-v1/DVSR1_validation_result.json'
for path in (CONFIRMATION_ARM,CONFIRMATION_DECISION,VALIDATION_CACHE):
    if not path.is_file(): raise FileNotFoundError(path)
audit=json.loads(TRAIN_AUDIT.read_text()); assert audit['decision']=='AUTHORIZE_DVSR1_VALIDATION_SCREEN'
assert audit['validation_images_accessed'] is False and audit['test_images_accessed'] is False
print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)


In [ ]:
import importlib, os, shutil, subprocess, sys
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/defectoscafeverde-dual-view-fusion'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_dvsr1_screen.py'],cwd=REPO,check=True)


In [ ]:
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_dvsr1_screen','--train-audit',str(TRAIN_AUDIT),'--validation-cache',str(VALIDATION_CACHE),'--confirmation-arm',str(CONFIRMATION_ARM),'--confirmation-decision',str(CONFIRMATION_DECISION),'--output',str(OUTPUT)]
print('MENJALANKAN FROZEN DVSR1 VALIDATION SCREEN')
subprocess.run(command,cwd=REPO,check=True)
result=json.loads(OUTPUT.read_text())
assert result['selector_fitting_executed'] is False
assert result['detector_training_executed'] is False and result['fuser_training_executed'] is False
assert result['test_images_accessed'] is False


In [ ]:
import pandas as pd
metrics=('physical_pair_accuracy','macro_class_accuracy','bottom3_class_accuracy','worst_class_accuracy')
rows=[]
for model in ('PAPER_MAX_CONFIDENCE','DVSR1','PAIR_ORACLE'):
    rows.append({'model':model,**{metric:result['values'][model][metric] for metric in metrics}})
rows.insert(2,{'model':'CONFIRMED_DVF1_MEAN',**result['confirmed_dvf1_mean']})
display(pd.DataFrame(rows).style.format({metric:'{:.2%}' for metric in metrics}))
class_rows=[]
paper=result['values']['PAPER_MAX_CONFIDENCE']['accuracy_by_class']; candidate=result['values']['DVSR1']['accuracy_by_class']
for name,value in candidate.items(): class_rows.append({'class_name':name,'paper':paper[name],'DVSR1':value,'delta':value-paper[name]})
display(pd.DataFrame(class_rows).sort_values('delta',ascending=False).style.format({'paper':'{:.2%}','DVSR1':'{:.2%}','delta':'{:+.2%}'}))
print('VS PAPER:',result['deltas_vs_paper'])
print('VS CONFIRMED DVF1 MEAN:',result['deltas_vs_confirmed_dvf1_mean'])
print('TRANSITIONS:',result['transitions_vs_paper'])
print('CRITERIA:',result['criteria'])
print('GATES:',result['gates'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('TRAINING:',result['selector_fitting_executed'] or result['detector_training_executed'] or result['fuser_training_executed'],'| TEST:',result['test_images_accessed'])
print('SUMMARY:',OUTPUT)
